# RVC 1 — Ch. 3.1: Paths and Trajectories

This notebook is the Python counterpart of **`03_rvc1_ch3.1_PathsTrajs.mlx`**.  It follows the same teaching sequence while using the current **Robotics Toolbox for Python** and **Spatial Math Toolbox for Python** APIs.

## Learning objectives

By the end of this notebook you should be able to:

- distinguish a **path** from a **trajectory**;
- generate smooth scalar trajectories with quintic polynomials;
- generate trapezoidal (linear-segment-with-parabolic-blend) trajectories;
- generate multi-segment trajectories through via points;
- generate synchronized multi-axis / joint-space trajectories;
- interpolate orientation using RPY coordinates and unit quaternions;
- generate full Cartesian trajectories that interpolate translation and rotation simultaneously.

> **MATLAB → Python naming note**
>
> - `tpoly(...)` → `quintic(...)`
> - `lspb(...)` → `trapezoidal(...)` (`lspb` is deprecated in the current Python toolbox)
> - `[q, qd, qdd] = ...` → a Python `Trajectory` object with `.q`, `.qd`, `.qdd` (aliases `.s`, `.sd`, `.sdd` also exist)

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from spatialmath import SE3, SO3, UnitQuaternion
from spatialmath.base import trplot
from roboticstoolbox import (
    quintic,
    trapezoidal,
    jtraj,
    mtraj,
    mstraj,
    ctraj,
)

np.set_printoptions(precision=4, suppress=True)

---
# Lec 04.1 — Paths and trajectories

Paths and trajectories are critical in robotics because they determine **how a robot moves**.

Typical design goals include:

- smoothness,
- energy efficiency,
- safety,
- accuracy.

A **path** describes the geometric route between poses.  It does **not** prescribe how quickly the robot moves along that route.  A **trajectory** adds a time law, so position, velocity and acceleration become functions of time.

We begin with a starting pose and a goal pose.

### Starting pose

The starting pose is the identity transform: zero translation and zero rotation.

In [ ]:
T0 = SE3()
T0

In [ ]:
T0.plot(frame='0', color='black', dims=[-1, 4, -1, 4, -1, 4])
plt.title('Starting pose $T_0$');

### Goal pose

The goal pose is translated by `(1, 2, 3)` and rotated by RPY angles `(0.6, 0.8, 1.4)` rad.

In [ ]:
T1 = SE3(1, 2, 3) * SE3.RPY([0.6, 0.8, 1.4])
T1

In [ ]:
T1.plot(frame='1', color='blue', dims=[-1, 4, -1, 4, -1, 4])
plt.title('Goal pose $T_1$');

### Visualizing both endpoint poses

In [ ]:
ax = T0.plot(frame='0', color='black', dims=[-1, 4, -1, 4, -1, 4])
T1.plot(frame='1', color='blue', ax=ax)
plt.title('Start and goal poses');

### Optional animation

Spatial Math pose objects can animate interpolation between poses.  Depending on the notebook backend, animations may open an external Matplotlib window.

In [ ]:
# Optional interactive animation:
# T0.animate(T1, frame='A', dims=[-1, 4, -1, 4, -1, 4])

---
# Lec 04.2 — Polynomial (smooth) trajectories

We begin with a simple **one-dimensional smooth trajectory**.

Suppose the coordinate starts at 0 and ends at 1.  In MATLAB RVC this example uses `tpoly`; the current Python toolbox calls the corresponding fifth-order polynomial generator **`quintic()`**.

A quintic polynomial is especially useful because it can satisfy endpoint constraints on position, velocity, and acceleration while remaining smooth.

In [ ]:
tg = quintic(0, 1, 50)
tg

In [ ]:
tg.plot();

Analyze the three profiles above:

1. Does the position vary smoothly from the initial to final value?
2. What are the initial and final velocities?
3. How does the trajectory accelerate and decelerate so that it meets both position and velocity boundary conditions smoothly?

### Changing the target

In [ ]:
tg_neg = quintic(0, -1, 50)
tg_neg.plot();

### Accessing position, velocity and acceleration

Unlike MATLAB, where multiple outputs are returned separately, the Python toolbox returns a `Trajectory` object.

In [ ]:
tg = quintic(0, 1, 50)

s = tg.q      # position
sd = tg.qd    # velocity
sdd = tg.qdd  # acceleration

print('position shape:', s.shape)
print('velocity shape:', sd.shape)
print('acceleration shape:', sdd.shape)
print('first 5 positions:', s[:5])

### Controlling initial and final velocities

We can explicitly prescribe the initial and final velocity constraints.

In [ ]:
tg_v = quintic(0, 1, 50, qd0=0.5, qdf=0)
tg_v.plot();

### Time steps versus physical time

Passing an **integer** such as `50` means 50 trajectory samples, with derivatives expressed per trajectory step.  If physical units such as m/s or rad/s matter, pass an explicit time vector.

In [ ]:
t = np.linspace(0, 5, 50)   # 5 seconds
tg_time = quintic(0, 1, t)
tg_time.plot();

---
# Lec 04.3 — 1D trapezoidal trajectories

A trapezoidal-velocity trajectory uses a **linear segment with parabolic blends**.  The velocity ramps up, remains approximately constant, and then ramps down.

This is called `lspb` in the MATLAB material.  In the current Python toolbox, use **`trapezoidal()`**.

In [ ]:
trap = trapezoidal(0, 1, 50)
trap.plot();

We can again retrieve position, velocity, and acceleration.

In [ ]:
s = trap.q
sd = trap.qd
sdd = trap.qdd

print('blend duration:', trap.tblend)
print('maximum velocity:', np.max(sd))

### Controlling the trapezoidal velocity

The optional `V` argument sets the velocity of the linear segment.

In [ ]:
trap_v1 = trapezoidal(0, 1, 50, V=0.025)
trap_v1.plot();

In [ ]:
trap_v2 = trapezoidal(0, 1, 50, V=0.035)
trap_v2.plot();

As the requested plateau velocity increases, less of the trajectory is spent in the constant-velocity phase.  The acceleration profile is piecewise constant and therefore changes discontinuously at the phase transitions.

### Infeasible velocity requests

Not every requested plateau velocity is compatible with the displacement and allotted time.  The toolbox raises an error when the requested profile is impossible.

In [ ]:
try:
    trapezoidal(0, 1, 50, V=0.02)
except ValueError as e:
    print('Infeasible request:', e)

If the trajectory is described using a physical time vector, changing the total time can make a velocity requirement feasible.

In [ ]:
t_long = np.linspace(0, 10, 100)
trap_long = trapezoidal(0, 1, t_long, V=0.20)
trap_long.plot();

---
# Lec 04.4 — 1D trajectories with via points

Robots often need to pass through intermediate **waypoints** or **via points**.

`mstraj()` generates a **multi-segment, multi-axis trajectory** consisting of linear segments connected by polynomial blends.

Current Python signature:

```python
mstraj(viapoints, dt, tacc,
       qdmax=None, tsegment=None,
       q0=None, qd0=None, qdf=None)
```

where:

- `viapoints`: one waypoint per row;
- `qdmax`: scalar or per-axis speed limit;
- `tsegment`: duration of each segment (alternative to `qdmax`);
- `q0`: initial coordinate;
- `dt`: sampling interval;
- `tacc`: acceleration/blend time.

In [ ]:
first = np.array([10.0])
last = 30.0
via_1d = np.array([[40.0], [10.0], [last]])

traj_via = mstraj(
    via_1d,
    dt=0.1,
    tacc=2,
    qdmax=1,
    q0=first,
)

print(traj_via)
print('arrival times:', traj_via.arrive)

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(traj_via.t, traj_via.q[:, 0], label='trajectory')
for y in [first[0], *via_1d[:, 0]]:
    plt.axhline(y, ls=':', alpha=0.35)
plt.xlabel('time (s)')
plt.ylabel('position')
plt.title('1D multi-segment trajectory')
plt.grid(True)
plt.legend();

### Increase the maximum velocity

Increasing `qdmax` allows segments to be traversed faster.

In [ ]:
traj_fast = mstraj(via_1d, dt=0.1, tacc=2, qdmax=2, q0=first)

print(f'qdmax=1 duration: {traj_via.t[-1]:.1f} s')
print(f'qdmax=2 duration: {traj_fast.t[-1]:.1f} s')

### Hard constraints on segment duration

Instead of supplying `qdmax`, we may directly provide a desired duration for each segment.

In [ ]:
tseg = [5, 4, 3]
traj_tseg = mstraj(
    via_1d,
    dt=0.1,
    tacc=1,
    tsegment=tseg,
    q0=first,
)

plt.plot(traj_tseg.t, traj_tseg.q[:, 0])
plt.xlabel('time (s)')
plt.ylabel('position')
plt.title('Specified segment durations: [5, 4, 3] s')
plt.grid(True);

With polynomial blends, the geometric via points need not be hit exactly.  The blend smooths the transition from one segment to the next.  This is an important distinction between **passing near a via point smoothly** and enforcing an exact stop at every point.

In [ ]:
tseg_relaxed = [10, 30, 10]
traj_relaxed = mstraj(
    via_1d,
    dt=0.1,
    tacc=2,
    tsegment=tseg_relaxed,
    q0=first,
)

plt.plot(traj_relaxed.t, traj_relaxed.q[:, 0])
plt.xlabel('time (s)')
plt.ylabel('position')
plt.title('Longer segment durations')
plt.grid(True);

### Effect of acceleration/blend time

A shorter `tacc` requires more aggressive changes in velocity.  A longer `tacc` produces gentler transitions, but the blend occupies more of each segment.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for tacc in [2, 4, 8]:
    tr = mstraj(via_1d, dt=0.1, tacc=tacc, tsegment=tseg_relaxed, q0=first)
    ax.plot(tr.t, tr.q[:, 0], label=f'tacc={tacc} s')
ax.set_xlabel('time (s)')
ax.set_ylabel('position')
ax.set_title('Effect of blend time')
ax.grid(True)
ax.legend();

---
# Lec 04.5 — Multidimensional smooth trajectories

We now move from a scalar coordinate to multiple synchronized dimensions.  These dimensions might be:

- Cartesian coordinates `(x, y, z)`, or
- robot joint coordinates `(q1, q2, ..., qn)`.

`jtraj()` generates a smooth **quintic joint-space trajectory** between two configuration vectors.

### Two-axis example

Imagine a two-joint robot whose joint coordinates move from `[1, 2]` rad to `[3, 1]` rad.

In [ ]:
q0 = np.array([1.0, 2.0])
qf = np.array([3.0, 1.0])

jt = jtraj(q0, qf, 50)
print(jt)

In [ ]:
plt.plot(jt.q)
plt.xlabel('trajectory step')
plt.ylabel('angle (rad)')
plt.title('Joint trajectory — position')
plt.grid(True)
plt.legend(['Joint 1', 'Joint 2']);

The quintic polynomial provides smooth position and velocity profiles with zero default endpoint velocity and acceleration.

In [ ]:
plt.plot(jt.qd)
plt.xlabel('trajectory step')
plt.ylabel('velocity (rad/step)')
plt.title('Joint trajectory — velocity')
plt.grid(True)
plt.legend(['Joint 1', 'Joint 2']);

### Specifying initial and final velocities

In [ ]:
jt_v = jtraj(
    q0,
    qf,
    50,
    qd0=[0, 0],
    qd1=[10, 10],
)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(jt_v.q)
ax[0].set(xlabel='trajectory step', ylabel='angle (rad)', title='Position')
ax[0].grid(True)
ax[0].legend(['Joint 1', 'Joint 2'])

ax[1].plot(jt_v.qd)
ax[1].set(xlabel='trajectory step', ylabel='velocity', title='Velocity')
ax[1].grid(True)
ax[1].legend(['Joint 1', 'Joint 2']);

> With an integer sample count, velocity is expressed **per trajectory step**.  Use an explicit physical time vector if `10 rad/s` is literally intended.

## Multiple segments, multiple dimensions

`jtraj()` joins one start configuration to one end configuration.  For multiple via points, return to `mstraj()`.

In [ ]:
start = np.array([0.4, 0.5])
via = np.array([
    [0.6, 0.3],
    [0.4, 0.1],
    [0.2, 0.3],
    start,
])

mt = mstraj(
    via,
    dt=0.1,
    tacc=1,
    qdmax=0.2,
    q0=start,
)

print('trajectory shape:', mt.q.shape)
print('duration:', mt.t[-1])

In [ ]:
plt.plot(mt.t, mt.q)
plt.xlabel('time (s)')
plt.ylabel('angle (rad)')
plt.title('Synchronized multi-joint trajectory')
plt.grid(True)
plt.legend(['Joint 1', 'Joint 2']);

Although the two coordinates travel through different values, `mstraj()` synchronizes the axes so that segment transitions occur together.

### Configuration-space map

Instead of plotting each coordinate against time, plot joint 1 against joint 2.  This gives the path through the two-dimensional configuration space.

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(mt.q[:, 0], mt.q[:, 1])
plt.scatter(via[:, 0], via[:, 1], marker='x', s=80, label='via points')
plt.scatter(start[0], start[1], marker='o', s=60, label='start')
plt.xlabel('Joint 1 (rad)')
plt.ylabel('Joint 2 (rad)')
plt.title('Joint 1 vs. Joint 2')
plt.axis('equal')
plt.grid(True)
plt.legend();

### Different axis speed limits

A vector `qdmax` specifies a different speed limit for each axis.  Segment timing is governed by whichever axis is limiting.

In [ ]:
mt_axis = mstraj(
    via,
    dt=0.1,
    tacc=1,
    qdmax=[0.1, 0.3],
    q0=start,
)

plt.plot(mt_axis.t, mt_axis.q)
plt.xlabel('time (s)')
plt.ylabel('angle (rad)')
plt.title('Different per-axis speed limits')
plt.grid(True)
plt.legend(['Joint 1', 'Joint 2']);

---
# Lec 04.6 — Rotational interpolation

## Roll–pitch–yaw interpolation

RPY angles are one coordinate representation of orientation.  Because they form a three-dimensional coordinate vector, we can interpolate the three angle coordinates using `jtraj()`.

Here we interpolate from `(0, 0, 0)` to `(-π/2, π/2, π/4)` rad.

In [ ]:
rpy_traj = jtraj(
    [0, 0, 0],
    [-np.pi/2, np.pi/2, np.pi/4],
    100,
)

plt.plot(rpy_traj.q)
plt.xlabel('trajectory step')
plt.ylabel('angle (rad)')
plt.title('RPY coordinate trajectory')
plt.grid(True)
plt.legend(['roll', 'pitch', 'yaw']);

Each RPY coordinate can be converted into a rotation object.  In Python, `SO3.RPY()` naturally accepts the full `N×3` trajectory and creates an `SO3` sequence.

In [ ]:
Rseq = SO3.RPY(rpy_traj.q)

print('number of orientations:', len(Rseq))
print('first rotation matrix:')
print(Rseq[0].R)
print('10th rotation matrix:')
print(Rseq[9].R)

### Optional animation of the RPY trajectory

In [ ]:
# Depending on the notebook backend this may open an external window:
# Rseq.animate(frame='R', dims=[-1, 1, -1, 1, -1, 1])

## Trapezoidal multi-axis interpolation with `mtraj()`

`mtraj()` generalizes multi-dimensional interpolation by accepting a scalar trajectory generator.  This lets us choose `trapezoidal` or `quintic` while using the same start and end vectors.

In [ ]:
Q0 = np.array([0.0, 0.0, 0.0])
QF = np.array([np.pi/4, np.pi/2, np.pi/3])

rpy_trap = mtraj(trapezoidal, Q0, QF, 50)

fig, ax = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
ax[0].plot(rpy_trap.q)
ax[0].set_ylabel('position (rad)')
ax[0].grid(True)
ax[1].plot(rpy_trap.qd)
ax[1].set_ylabel('velocity')
ax[1].grid(True)
ax[2].plot(rpy_trap.qdd)
ax[2].set_ylabel('acceleration')
ax[2].set_xlabel('trajectory step')
ax[2].grid(True)
fig.suptitle('Multi-axis trapezoidal trajectory');

## Quaternion interpolation — SLERP

Unit quaternions are especially useful for rotational interpolation because **spherical linear interpolation (SLERP)** follows a geodesic on the unit-quaternion sphere and produces uniform angular interpolation for a linear interpolation parameter.

Start with the identity orientation and end with a 90° rotation about the x-axis.

In [ ]:
q1 = UnitQuaternion()
q2 = UnitQuaternion.Rx(np.pi/2)

print('start:', q1)
print('end:  ', q2)

The interpolation parameter `s` varies from 0 to 1.

In [ ]:
print('s=0.0:', q1.interp(q2, 0.0))
print('s=0.5:', q1.interp(q2, 0.5))
print('s=1.0:', q1.interp(q2, 1.0))

### Quaternion trajectory

In [ ]:
s_linear = np.linspace(0, 1, 51)
qseq = q1.interp(q2, s_linear)

print('number of quaternion samples:', len(qseq))
print('first:', qseq[0])
print('last: ', qseq[-1])

A useful non-animated diagnostic is to convert the quaternion sequence to rotation angles.

In [ ]:
angles = np.array([q.angvec()[0] for q in qseq])

plt.plot(s_linear, angles)
plt.xlabel('interpolation fraction $s$')
plt.ylabel('rotation angle (rad)')
plt.title('SLERP with a linear interpolation parameter')
plt.grid(True);

### Optional quaternion animation

In [ ]:
# Optional interactive animation:
# qseq.animate(frame='Q', dims=[-1, 1, -1, 1, -1, 1])

### Smooth time-scaling of SLERP

SLERP specifies the geometric interpolation of orientation.  We can independently choose a smooth **time law** for the interpolation parameter `s`.

For example, use a trapezoidal trajectory from 0 to 1 and feed that scalar profile to quaternion interpolation.

In [ ]:
s_profile = trapezoidal(0, 1, 50)
qseq_smooth = q1.interp(q2, s_profile.q)

fig, ax = plt.subplots(2, 1, figsize=(8, 6))
ax[0].plot(s_profile.q)
ax[0].set_ylabel('s')
ax[0].set_title('Trapezoidal time-scaling')
ax[0].grid(True)

angles_smooth = np.array([q.angvec()[0] for q in qseq_smooth])
ax[1].plot(angles_smooth)
ax[1].set_xlabel('trajectory step')
ax[1].set_ylabel('rotation angle (rad)')
ax[1].grid(True);

---
# Lec 04.7 — Cartesian interpolation

We now interpolate **translation and rotation simultaneously**.

A full Cartesian pose belongs to `SE(3)`.  We should not linearly interpolate individual entries of a homogeneous transformation matrix because its rotation block must remain orthonormal.  `ctraj()` handles this correctly:

- translation is interpolated in Euclidean space;
- orientation is interpolated using unit-quaternion interpolation.

The result is a sequence of valid `SE3` poses.

In [ ]:
T0 = SE3()
T1 = SE3(0, 5, 3) * SE3.RPY([0, 0, np.pi/2])

T = ctraj(T0, T1, 50)

print('number of poses:', len(T))
print('first pose:')
print(T[0])
print('final pose:')
print(T[-1])

### Plot the Cartesian path

In [ ]:
xyz = np.vstack([Ti.t for Ti in T])

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(xyz[:, 0], xyz[:, 1], xyz[:, 2], '-o', markersize=2)
ax.scatter(*T0.t, s=60, label='start')
ax.scatter(*T1.t, s=60, label='goal')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_title('Cartesian trajectory — translation component')
ax.legend();

### Inspect how translation and orientation evolve together

In [ ]:
rpy = np.vstack([Ti.rpy() for Ti in T])

fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
ax[0].plot(xyz)
ax[0].set_ylabel('translation')
ax[0].legend(['x', 'y', 'z'])
ax[0].grid(True)

ax[1].plot(rpy)
ax[1].set_xlabel('trajectory step')
ax[1].set_ylabel('RPY (rad)')
ax[1].legend(['roll', 'pitch', 'yaw'])
ax[1].grid(True)
fig.suptitle('Cartesian trajectory components');

### Optional Cartesian animation

In [ ]:
# Optional interactive animation:
# T.animate(frame='T', dims=[-1, 2, -1, 6, -1, 4])

---
# Summary: MATLAB → Python trajectory calls

| Concept | MATLAB RVC | Current Python toolbox |
|---|---|---|
| Quintic scalar trajectory | `tpoly(q0,qf,N)` | `quintic(q0,qf,N)` |
| Trapezoidal scalar trajectory | `lspb(q0,qf,N)` | `trapezoidal(q0,qf,N)` |
| Access trajectory outputs | `[s,sd,sdd] = ...` | `tg.q`, `tg.qd`, `tg.qdd` |
| Joint/multi-axis quintic | `jtraj(q0,qf,N)` | `jtraj(q0,qf,N)` |
| Generic multi-axis trajectory | `mtraj(@lspb,...)` | `mtraj(trapezoidal,...)` |
| Multi-segment trajectory | `mstraj(via,qdmax,tseg,q0,dt,tacc)` | `mstraj(via, dt, tacc, qdmax=..., tsegment=..., q0=...)` |
| Quaternion interpolation | `q1.interp(q2,s)` | `q1.interp(q2,s)` |
| Cartesian interpolation | `ctraj(T0,T1,N)` | `ctraj(T0,T1,N)` with `SE3` objects |

## Central idea

A trajectory can be understood as the combination of two related ideas:

1. **Geometry:** what path should the robot follow through configuration or Cartesian space?
2. **Timing:** how should the robot progress along that path while satisfying velocity and acceleration requirements?

This separation becomes increasingly important in motion planning, robot control, and optimization.

## References

- Peter Corke, *Robotics, Vision & Control for Python*, Springer, 2023, Chapter 3.
- Robotics Toolbox for Python trajectory documentation: https://petercorke.github.io/robotics-toolbox-python/arm_trajectory.html
- Spatial Math Toolbox for Python: https://bdaiinstitute.github.io/spatialmath-python/